In [4]:
import numpy as np
import pandas as pd
import random
import yfinance as yf

# 1. Download Historical Stock Data
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
# Use auto_adjust=True (default) and access the 'Close' column
data = yf.download(tickers, start='2023-01-01', end='2024-01-01')['Close']

# 2. Compute Daily Returns, Mean Returns, and Covariance
returns = data.pct_change().dropna()
mean_returns = returns.mean() * 252  # Annualized expected returns
cov_matrix = returns.cov() * 252     # Annualized covariance matrix

print("Annualized Expected Returns:")
print(mean_returns)
print("\nAnnualized Covariance Matrix:")
print(cov_matrix)



# 3. Particle Swarm Optimization (PSO) for Portfolio Optimization
NUM_ASSETS = len(tickers)
NUM_PARTICLES = 50
ITERATIONS = 100
RISK_FREE_RATE = 0.02

class PortfolioParticle:
    def __init__(self):
        # Initialize random weights
        raw_weights = np.random.rand(NUM_ASSETS)
        self.position = raw_weights / np.sum(raw_weights)  # Normalized so sum is 1
        self.velocity = np.zeros(NUM_ASSETS)
        self.best_position = np.copy(self.position)
        self.best_sharpe = self.evaluate(self.position)

    def evaluate(self, weights):
        # Portfolio Expected Return
        port_return = np.sum(mean_returns * weights)
        # Portfolio Expected Volatility
        port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        # Maximize Sharpe Ratio
        sharpe_ratio = (port_return - RISK_FREE_RATE) / (port_vol + 1e-8)
        return sharpe_ratio

    def update_position():
        pass # Updated inside main loop

# Initialize Swarm
swarm = [PortfolioParticle() for _ in range(NUM_PARTICLES)]
g_best_pos = np.copy(swarm[0].position)
g_best_sharpe = swarm[0].best_sharpe

for p in swarm:
    if p.best_sharpe > g_best_sharpe:
        g_best_sharpe = p.best_sharpe
        g_best_pos = np.copy(p.best_position)

# Optimization Loop
for it in range(ITERATIONS):
    for p in swarm:
        r1, r2 = random.random(), random.random()
        # Cognitive and Social velocity update
        cog = 1.5 * r1 * (p.best_position - p.position)
        soc = 1.5 * r2 * (g_best_pos - p.position)
        p.velocity = 0.5 * p.velocity + cog + soc

        # Update weights and normalize to enforce sum = 1 constraint
        new_position = p.position + p.velocity
        new_position = np.clip(new_position, 0.0, 1.0) # No short selling constraint

        # If all weights became 0, reinitialize randomly
        if np.sum(new_position) == 0:
            new_position = np.random.rand(NUM_ASSETS)

        p.position = new_position / np.sum(new_position) # Normalize weight sum to 1

        # Evaluate fitness
        current_sharpe = p.evaluate(p.position)
        if current_sharpe > p.best_sharpe:
            p.best_sharpe = current_sharpe
            p.best_position = np.copy(p.position)

        if current_sharpe > g_best_sharpe:
            g_best_sharpe = current_sharpe
            g_best_pos = np.copy(p.position)

print("\n--- PSO Portfolio Optimization Complete ---")
print(f"Optimized Sharpe Ratio: {g_best_sharpe:.4f}")
for ticker, weight in zip(tickers, g_best_pos):
    print(f"  {ticker}: {weight * 100:.2f}%")
print(f"Total Weight Sum: {np.sum(g_best_pos) * 100:.1f}%")

/tmp/ipykernel_1512/972638662.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start='2023-01-01', end='2024-01-01')['Close']
[*********************100%***********************]  4 of 4 completed


Annualized Expected Returns:
Ticker
AAPL     0.462402
AMZN     0.632897
GOOGL    0.501407
MSFT     0.496859
dtype: float64

Annualized Covariance Matrix:
Ticker      AAPL      AMZN     GOOGL      MSFT
Ticker                                        
AAPL    0.039818  0.029108  0.032044  0.027468
AMZN    0.029108  0.109076  0.060323  0.047781
GOOGL   0.032044  0.060323  0.092405  0.038936
MSFT    0.027468  0.047781  0.038936  0.063103

--- PSO Portfolio Optimization Complete ---
Optimized Sharpe Ratio: 2.4601
  AAPL: 61.23%
  MSFT: 20.64%
  GOOGL: 0.00%
  AMZN: 18.13%
Total Weight Sum: 100.0%
